# Lesson 26 · 🎨 Calculus visualized — seeing the walk downhill: tangent lines, contour lines and gradient-descent trajectories

> **🎨 How to read this**: the picture-book edition of L22–L25. The narration stays short — **watch in the plots how a step size that is too large oscillates, and how contour lines and the negative gradient point to the bottom of the valley**. No need to re-teach all of gradient descent.

← **Previous lesson** · [L25 · Gradient descent](L25_gradient_descent.ipynb)

> Last lesson covered **gradient descent**: walking downhill blindfolded, following the negative gradient to the bottom of the valley, from loss function to weight-update rule.
> This lesson explores **calculus visualized**.

## This lesson's story: drawing the slope

The slope of a tangent line tells you which way a function is changing at a point; the gradient generalizes that direction to many dimensions; and gradient descent walks, step by step, against the gradient toward the lowest point. This lesson draws one picture for each of the three with `tangent`, `descent_1d` and `contour_descent`, then uses `lr_compare` to compare how three learning rates (lr) converge.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from cviz import style, tangent, descent_1d, contour_descent, lr_compare
style(); print('cviz ready')

## Into calculus visualized: starting from a numerical approximation of the derivative

The derivative $f'(x)$ is an **instantaneous rate of change**. In L22 we learned:
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

That limit is beautiful in theory, but a computer can't actually take a limit. We approximate it with "a small but finite h" — this is **numerical differentiation**.

**A natural question**: if the definition of the derivative uses $f(x+h) - f(x)$, why do we so often use $f(x+h) - f(x-h)$ instead?

The plainest answer: it's more symmetric. Looking at the curve $y = f(x)$ near the point $x$ from the left and from the right at the same time lets the two sides cancel out some of each other's error. That is the idea behind the **central difference**. In mathematical language: the one-sided difference has error of order $O(h)$, while the central difference brings it down to $O(h^2)$ — so as the step h shrinks, the accuracy improves much faster.

## Into the lab: watching a function through its numerical changes

Use the central difference (CD) `(f(x+h)−f(x−h))/(2h)` to compute the approximate slope of `f(x) = x²` at five points, and compare it row by row with the theoretical value `2x`.

## Hands-on: rate of change is more than a slogan

For `f(x) = x²` with `h = 1e-3`, the central difference matches the theoretical value `2x` to three decimal places.

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('approximate slope =', np.round(slopes, 3))
print('theoretical slope 2x =', 2 * xs)


### Why 2h and not h?

Look at the code above: the numerator is $f(x+h) - f(x-h)$. How far apart are those two points?
- The distance from $(x-h)$ to $(x+h)$ is $(x+h) - (x-h) = 2h$

So the denominator of the rate of change (change in height ÷ horizontal distance) has to be $2h$, not $h$. It's like a mountain path: walk h to the left and h to the right, and the total distance between the two ends is $2h$.

One more time, the formula:
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

## Exercise: implement the central difference and check its accuracy

**Task**: fill in `central_diff(f, x, h)`, then check, for `f(x) = x²` at `x = 1.5`:
- the error between the numerical derivative and the theoretical value `2 x 1.5 = 3.0` is below `1e-6`
- one negative-gradient update strictly lowers the function value: `f(x - lr*slope) < f(x)`

When you're done, run the assertions; if they pass, the cell prints "Check passed".

In [ ]:
def central_diff(f, x, h=1e-5):
    """Approximate the first derivative of f at x with a central difference.
    Parameters:
        f : callable, takes a scalar and returns a scalar
        x : float, the point to differentiate at
        h : float, step size (default 1e-5)
    Returns: float
    """
    raise NotImplementedError("Implement the central difference formula: (f(x+h) - f(x-h)) / (2*h)")

# --- check area ---
try:
    import numpy as np
    f_sq = lambda x: x**2
    x0, lr = 1.5, 0.1
    slope = central_diff(f_sq, x0)
    assert abs(slope - 2 * x0) < 1e-6, f'error too large: {abs(slope - 2*x0):.2e} — check that the denominator is 2h'
    x_new = x0 - lr * slope
    assert f_sq(x_new) < f_sq(x0), 'f did not decrease after one negative-gradient step — check that the sign is a minus'
    print('Check passed: central-difference error', f'{abs(slope - 2*x0):.2e}', ' function value', f_sq(x0), '->', f'{f_sq(x_new):.4f}')
except (NotImplementedError, TypeError) as e:
    print(f'[TODO] {e}')


### Why does a "negative-gradient update" always lower the function value?

The check code tested `f(x - lr*slope) < f(x)` — that "a small step in the negative-gradient direction" lowers the function value. That's no coincidence; the maths guarantees it.

**What the Taylor expansion tells us**: near the point $x$, take the first-order Taylor approximation:
$$f(x + \Delta x) \approx f(x) + f'(x) \cdot \Delta x$$

If we choose $\Delta x = -\text{lr} \cdot f'(x)$ (that is, we walk in the negative-gradient direction), then:
$$f(x - \text{lr} \cdot f'(x)) \approx f(x) - \text{lr} \cdot (f'(x))^2$$

The key observation: $(f'(x))^2 \geq 0$ is always non-negative (it's a square). So:
$$f(x - \text{lr} \cdot f'(x)) \approx f(x) - \text{lr} \cdot (\text{non-negative}) \leq f(x)$$

When $f'(x) \neq 0$ (that is, we're not at an extremum), the inequality is a strict $<$ and the function value strictly decreases.

**The condition**: this argument assumes the step size $\text{lr}$ is small enough for the first-order Taylor approximation to be accurate. If $\text{lr}$ is too large, the higher-order terms kick in and can break the descent property. Which brings us back, once again, to "balancing the learning rate".

### ⚠️ Choosing h: smaller is not always better

Beginners often ask: "If h matters so much, what about h=1e-10 or 1e-20 — won't that make the error even smaller?"

**No, it won't.** There's a hidden trap: **floating-point precision**.

Computers store floating-point numbers in binary with limited precision (a 64-bit float has about 15–16 significant digits). When h is too small:
- $f(x+h)$ and $f(x-h)$ come out extremely close (both roughly equal to $f(x)$)
- subtracting two nearly equal large numbers destroys significant digits and introduces **rounding error** instead

An analogy: you have a scale accurate to "5 decimal places", and two heavy weights both read 100.00000 kg. Any estimate you build from their difference is meaningless.

**The practical choice**:
- h = 1e-5 to 1e-7 is usually a safe balance point: small enough for the central-difference formula to be accurate, large enough to avoid rounding error
- for a polynomial like f(x)=x², h=1e-5 gives an error of 1e-8 or better
- if the function is more complicated (e.g. has large higher-order derivatives), you may need a slightly larger h

## Code experiment: walk across positions and watch the slope change

For `f(x) = x² + 2x`, print the function value and slope at each point from `x = −3` to `x = 3`, and watch the slope cross zero near `x = −1`.

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


### A side note: where does the derivative of $f(x) = x^2$ come from?

The code above uses `f(x) = x² + 2x`. What is its derivative? Let's work it out from the limit definition, looking only at the $x^2$ part:

$$f'(x) = \lim_{h \to 0} \frac{(x+h)^2 - x^2}{h}$$

Expand $(x+h)^2 = x^2 + 2xh + h^2$:

$$f'(x) = \lim_{h \to 0} \frac{x^2 + 2xh + h^2 - x^2}{h} = \lim_{h \to 0} \frac{2xh + h^2}{h} = \lim_{h \to 0} (2x + h)$$

As $h \to 0$, $2x + h \to 2x$, so $(x^2)' = 2x$.

And $2x$? Same method: $\frac{d}{dx}(2x) = 2$ (the derivative of a constant multiple of x is just that multiple).

So the derivative of $f(x) = x^2 + 2x$ is $f'(x) = 2x + 2$. This is the "differentiate term by term" rule: $(u + v)' = u' + v'$.

## 1. Derivative = slope of the tangent (goes with L22)

`tangent(f, grad_f, x0)` draws the function's curve and its tangent line at the point `x0`; the tangent's slope is `f'(x0)`. For `f(x) = x^2`, the tangent at `x0 = 1.5` has slope `3.0`; in the negative region the slope is negative; and at the vertex `x0 = 0` the slope is exactly zero — the minimum is where the gradient goes to zero.

This picture is the geometric foundation of the whole lesson. Visualizing gradient descent comes down to two key pictures: in one dimension, it's sliding down the curve, each step moving against the derivative; in two dimensions, it's moving toward lower ground perpendicular to the contour lines, each step always at right angles to the contour. The effect of the learning rate is visible at a glance: with lr too large the trajectory bounces left and right across the valley floor; with a moderate lr it spirals in to the minimum. Once you clearly see that, in one dimension, the slope gives you the downhill direction, the multi-dimensional version — the gradient is the direction of steepest ascent, and you go the opposite way — is easy to follow.

### The tangent line's equation, concretely

At the point $(x_0, f(x_0))$, the tangent is a straight line. Write it in "point–slope form":
$$y - f(x_0) = f'(x_0) \cdot (x - x_0)$$

Rearranged:
$$y = f(x_0) + f'(x_0) \cdot (x - x_0)$$

**A concrete example**: for $f(x) = x^2$ at $x_0 = 1.5$:
- function value: $f(1.5) = 2.25$
- derivative: $f'(1.5) = 2 \times 1.5 = 3.0$
- tangent line: $y = 2.25 + 3.0 \times (x - 1.5) = 3.0x - 2.25$

On the plot, this line touches the curve at $(1.5, 2.25)$ with slope 3.0. Extend it in both directions and it seems to "hug" the parabola, rather than cutting through it or floating above it. That is the geometric meaning of "tangent" (from the Latin *tangere*, "to touch").

In [ ]:
tangent(lambda x: x**2, lambda x: 2*x, x0=1.5);

## 2. Gradient descent in 1-D: rolling down the slope to the lowest point (goes with L25)

## From tangent to gradient descent: walking in the downhill direction

**Key observation**: the sign of the derivative tells you which way the function is heading, up or down.

For $f(x) = x^2$:
- at $x = -2$, $f'(-2) = 2 \times (-2) = -4 < 0$: moving right decreases the function value (downhill)
- at $x = 2$, $f'(2) = 2 \times 2 = 4 > 0$: moving right increases the function value (uphill)
- at $x = 0$, $f'(0) = 0$: this is the bottom (flat ground)

**The core rule of gradient descent**: take a small step in the direction of **minus the derivative**.
$$x_{\text{new}} = x - \text{learning rate} \times f'(x)$$

Why the minus sign? Because:
- if $f'(x) > 0$ (uphill), then $-f'(x) < 0$, and we move left (decrease x)
- if $f'(x) < 0$ (downhill), then $-f'(x) > 0$, and we move right (increase x)

Wherever you are, the "$-f'(x)$ direction" always points toward lower function values. That's what "the direction of steepest descent" means.

In [ ]:
descent_1d(lambda x:(x)**2, lambda x:2*x, x0=-2.5, lr=0.1);

## 3. Gradient descent in 2-D: downhill, perpendicular to the contours (goes with L23 gradients + L25)

**Aurora**: the zigzag line drawn by `contour_descent(start, lr, steps)` is a miniature of how a neural network's parameters get updated. Aurora's audio loss functions (such as a spectral reconstruction loss) will be implemented in later milestones; when they are, the update rule at each step is exactly the same as here (`x ← x − lr·∇f`) — only the parameter space grows from two dimensions to millions, and the zigzag becomes a trajectory through an extremely high-dimensional space.

### Partial derivatives: "directional derivatives" with many variables

Going from one dimension to many, the core idea stays the same, but now we have to ask: "in a given direction, how fast does the function change?"

For a function $f(x,y)$ we can ask two questions:
1. **Hold y fixed, change only x**: the **partial derivative** of $f$ with respect to $x$, $\frac{\partial f}{\partial x}$
2. **Hold x fixed, change only y**: the **partial derivative** of $f$ with respect to $y$, $\frac{\partial f}{\partial y}$

**How to compute it**: treat the other variable as a constant and differentiate.

**Example**: $f(x,y) = x^2 + 2y^2$
- $\frac{\partial f}{\partial x} = 2x$ (treat $2y^2$ as a constant; $(x^2)' = 2x$)
- $\frac{\partial f}{\partial y} = 4y$ (treat $x^2$ as a constant; $(2y^2)' = 4y$)

At the point $(x,y) = (1,2)$:
- $\frac{\partial f}{\partial x}\bigg|_{(1,2)} = 2 \times 1 = 2$
- $\frac{\partial f}{\partial y}\bigg|_{(1,2)} = 4 \times 2 = 8$

### The gradient vector: the direction of steepest ascent

Pack the two partial derivatives into one vector and you have the **gradient**:
$$\nabla f = \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right)$$

Geometrically: at any point, $\nabla f$ points in **the direction in which the function grows fastest**. Its opposite, $-\nabla f$, points in **the direction in which the function falls fastest** (the steepest way downhill).

For $f(x,y) = x^2 + 2y^2$ at $(1,2)$: $\nabla f = (2, 8)$

This vector points up and to the right, meaning: starting from $(1,2)$, heading up-right increases $f$ fastest. Conversely, heading down-left decreases $f$ fastest.

### Contour lines: the path traced by points at the same height

Imagine standing on a mountain and looking down. Contour lines are the curves on a map that **connect points of the same height**.

For $f(x,y)$, every point $(x,y)$ on contour $c$ satisfies $f(x,y) = c$ (a constant).

**The key geometric fact**: **the gradient is perpendicular to the contour lines**.

Why? Because every point on a contour has the same function value. The gradient is "the direction in which the function changes fastest", while along the contour's **tangent direction** the function doesn't change at all (same height). So the gradient must be perpendicular to that tangent — that is, perpendicular to the contour line itself.

**Picturing gradient descent**: every step goes along $-\nabla f$ — perpendicular to the contours, toward the lower side.

In [ ]:
contour_descent(start=(-2.3, 2.0), lr=0.15);

## 4. Effect of the learning rate: too small is slow / just right / too large oscillates

In [ ]:
lr_compare();

In [ ]:
# Parameter experiment 1: start right next to the minimum, lr=0.5, watch it oscillate
contour_descent(start=(0.1, 0.1), lr=0.5)
# Expected: f(x,y)=x²+2y² is convex and (0,0) is the global minimum; with a large lr the trajectory bounces back and forth around the minimum


## Parameter experiment: starting right next to the minimum

In `contour_descent`, change `start` to `(0.1, 0.1)` so the trajectory starts right next to the minimum. Note: `f(x,y)=x²+2y²` is a **convex function**, and (0,0) is its global minimum, not a saddle point. A real saddle point needs the function to rise in one direction and fall in another, as in `f(x,y)=x²−y²`. Gradient descent on a convex function (with a small enough learning rate) is bound to converge; saddle points only show up with the non-convex losses of deep learning.

### Convex functions: definition and how to recognize one

**Intuitive definition**: if the graph of a function is shaped like a "bowl" or a "valley", it's convex. More precisely: the line segment between any two points on the graph lies on or above the graph.

**Mathematical definition**: for any two points $x_1, x_2$ and any $0 \leq t \leq 1$:
$$f(t \cdot x_1 + (1-t) \cdot x_2) \leq t \cdot f(x_1) + (1-t) \cdot f(x_2)$$

**How to tell** (for functions whose second derivative exists):
- **1-D**: $f''(x) \geq 0$ for all $x$ $\Rightarrow$ convex
- **2-D**: the Hessian matrix (the matrix of second partial derivatives) is **positive semidefinite**

**Examples**:
- $f(x) = x^2$ has second derivative $f''(x) = 2 > 0$, so it's convex
- $f(x,y) = x^2 + 2y^2$ has Hessian $\begin{pmatrix} 2 & 0 \\ 0 & 4 \end{pmatrix}$ (both eigenvalues positive), so it's convex

**What gradient descent is guaranteed**: for a convex function, **every local minimum is a global minimum**, so gradient descent (with a small enough learning rate) is bound to converge to the true optimum.

### Saddle points: neither a peak nor a valley

**Intuitive definition**: at this point the function increases in some directions (like a ridge) and decreases in others (like a valley). Around the point the surface looks like a saddle.

**Example**: $f(x,y) = x^2 - y^2$ at the origin $(0,0)$
- along the $x$ axis: $f(x,0) = x^2$, a valley ($x=0$ is a minimum)
- along the $y$ axis: $f(0,y) = -y^2$, a peak ($y=0$ is a maximum)

This function's Hessian is $\begin{pmatrix} 2 & 0 \\ 0 & -2 \end{pmatrix}$ (one positive eigenvalue, one negative), so it has a saddle point.

**Gradient descent's dilemma**: near a saddle point the gradient can be tiny (so the updates crawl), or the algorithm can rock back and forth in different directions and struggle to escape. This is the central difficulty of **non-convex optimization**.

Comparison experiment: keep `start=(0.1, 0.1)`, change `lr` from `0.15` to `0.5`, and watch whether the trajectory bounces back and forth around the minimum; then lower `lr` to `0.05` and see how the number of steps to converge goes up.

In [ ]:
# Comparison experiment: same start, lr=0.05, watch it converge slowly
contour_descent(start=(0.1, 0.1), lr=0.05)
# Expected: each step is small and the path heads steadily for the minimum, taking far more steps than lr=0.5


### The learning rate's convergence condition: when does it diverge?

The gradient-descent update rule is $x_{\text{new}} = x - \text{lr} \cdot f'(x)$. When does this iteration diverge?

**The 1-D case**: for $f(x) = x^2$, the derivative is $f'(x) = 2x$, so the update becomes:
$$x_{\text{new}} = x - \text{lr} \cdot 2x = (1 - 2 \cdot \text{lr}) \cdot x$$

Each iteration multiplies $x$ by the factor $(1 - 2 \cdot \text{lr})$. For $x$ to head to 0, that factor's absolute value must be $< 1$:
$$|1 - 2 \cdot \text{lr}| < 1$$

Unpack the inequality:
$$-1 < 1 - 2 \cdot \text{lr} < 1$$
$$-2 < -2 \cdot \text{lr} < 0$$
$$0 < \text{lr} < 1$$

**Conclusion**: for $f(x) = x^2$, the learning rate must satisfy $0 < \text{lr} < 1$ to converge.
- $\text{lr} = 0.1$ or $0.5$: both in range, so it converges
- $\text{lr} = 1.1 > 1$: diverges (the absolute value of $x$ grows every step)
- $\text{lr} = -0.5 < 0$: diverges (the absolute value of $x$ grows every step)

**Generalizing to 2-D**: the condition gets more complicated (it involves second-derivative information about the objective), but the core idea is the same: a learning rate that is too large destroys stability.

## Wrap-up

You can now draw the tangent at any point with `tangent`, visualize every step of gradient descent with `descent_1d` and `contour_descent`, and compare how three learning rates converge side by side with `lr_compare`. This zigzag trajectory is the geometric source of the `loss` curve you'll see when Aurora's models train: the loss at each training step (iteration) corresponds to one position in parameter space, and what gets printed is that position's height on the loss axis. `L25` already applied the same update rule, `x ← x − lr·∇f(x)`, to the MSE loss of a line fit — exactly the same form.

### How the gradient-descent trajectory maps to the loss curve

Picture yourself iterating in a **parameter space**:
- **the horizontal plane**: two parameters $x$ and $y$ (such as the slope and intercept in a line fit)
- **the vertical axis**: the loss function $f(x,y)$ (such as MSE)

**The gradient-descent trajectory** (the zigzag): in the horizontal plane, it starts from some point and moves along $-\nabla f$ at each step, eventually spiraling in to the lowest point $(0,0)$.

**The loss curve** (a 1-D curve): take the trajectory points $(x_t, y_t)$ step by step, line up their loss values $f(x_t, y_t)$ as a sequence, and you have the curve of loss falling over the epochs.

**How the two connect**:
- the zigzag's projection onto the horizontal plane is the parameters' update history
- the loss curve is a side view of that zigzag: it reads off the height at each position

For example, if the first steps of the trajectory are $(2,2) \to (1,1) \to (0.5,0.5) \to \ldots$, then for $f = x^2 + 2y^2$ the loss values are $12 \to 3 \to 0.75 \to \ldots$, and those plot as a falling loss curve.

In [ ]:
import numpy as np

# Assertions on the maths behind the calculus visuals (stand-alone: needs neither cviz nor your implementation)

# 1. Central-difference accuracy: f(x)=x² at several points, error should be < 1e-8
f = lambda x: x**2
h = 1e-5
for xv in [-2.0, -1.0, 0.0, 1.5, 3.0]:
    cd = (f(xv + h) - f(xv - h)) / (2 * h)
    analytical = 2 * xv
    err = abs(cd - analytical)
    assert err < 1e-8, f"x={xv}: CD error {err:.2e} too large"
print("Central difference ✅  CD error of f(x)=x² below 1e-8 at all 5 points")

# 2. 2-D gradient: f(x,y)=x²+2y², ∇f=(2x,4y), equal to [2,8] at (1,2)
f2d = lambda p: p[0]**2 + 2 * p[1]**2
grad_f2d = lambda p: np.array([2 * p[0], 4 * p[1]])
p0 = np.array([1.0, 2.0])
g_analytical = grad_f2d(p0)
assert np.allclose(g_analytical, [2., 8.], atol=1e-12)
# numerical gradient
g_num = np.zeros(2)
hh = 1e-6
for i in range(2):
    ev = np.zeros(2); ev[i] = hh
    g_num[i] = (f2d(p0 + ev) - f2d(p0 - ev)) / (2 * hh)
assert np.allclose(g_num, g_analytical, atol=1e-8)
print(f"2-D gradient ✅  ∇f(1,2)={g_analytical}, numerical={np.round(g_num,6)}")

# 3. GD convergence: from x=-2.5 with lr=0.1, x should be close to 0 after 50 steps
f1d = lambda x: x**2
grad_f1d = lambda x: 2 * x
x = -2.5; lr = 0.1
for _ in range(50):
    x = x - lr * grad_f1d(x)
assert abs(x) < 1e-4, f"GD did not converge to 0, got {x}"
print(f"GD converges ✅  after 50 steps x={x:.6f}≈0  (lr=0.1, f(x)=x², start x=-2.5)")

# 4. Divergence condition: lr=1.1 > 1, (x-3)² does not converge
grad_f4 = lambda x: 2 * (x - 3)
x4 = 0.0; lr4 = 1.1
trajectory = [x4]
for _ in range(10):
    x4 = x4 - lr4 * grad_f4(x4)
    trajectory.append(x4)
assert abs(trajectory[-1]) > abs(trajectory[0]), "should diverge with lr=1.1"
print(f"Divergence ✅  lr=1.1, |x| along the trajectory grows: {[round(v,1) for v in trajectory[:5]]}...")
print("\nAll maths properties behind the calculus visuals check out ✅")

In [ ]:
# ✏️ Self-review for this lesson
l26_review = {
    "tangent_visualized":        None,  # can you explain tangent = local linear approximation? True/False
    "descent_trajectory_seen":   None,  # have you looked at the descent/contour trajectory plots? True/False
    "lr_compare_understood":     None,  # can you compare convergence/divergence for different lr? True/False
    "l25_connection":            None,  # see that it's the same update rule as L25's gd_step? True/False
}

unfilled = [k for k, v in l26_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l26_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L26 fully cleared! On to L27: probability basics')


---
⬆️ **Final check**: once you've read the wrap-up, run the maths-property check cell above, then fill in the self-review.


---

→ **Next lesson** · [L27 · Probability basics](../4_probability/L27_probability_basics.ipynb)

> Next lesson covers **probability basics**: events, conditional probability, independence and the law of large numbers.